# Multislice HRTEM Simulation: From Atoms to Focal Series

A high-resolution TEM image of anything thicker than a few ångströms is the result of the
electron being scattered many times inside the specimen. This tutorial builds such images with
PyTex's multislice engine, `pytex.diffraction.multislice`, which implements the algorithm of
abTEM step for step, and works through the cases an HRTEM user meets:

1. the atomic potential and how a crystal is cut into slices;
2. why a crystal must sit in an exactly periodic box;
3. sampling and the band limit;
4. the exit wave and how it evolves with thickness (channelling and Pendellösung);
5. the electron diffraction pattern, and what beam tilt does to it;
6. HRTEM images under uncorrected, Cs-corrected and negative-Cs optics;
7. **focal series** from a single exit wave;
8. the defocus–thickness map used to match experiments;
9. partial temporal coherence: Frank's envelope versus focal integration;
10. thermal diffuse scattering with frozen phonons;
11. other specimens: hexagonal titanium, a missing column, amorphous carbon and Thon rings;
12. a check against Bloch waves.

The derivations are in {doc}`../../theory/multislice_hrtem`, the algorithm in
{doc}`../../algorithms/multislice_hrtem`, and the verified numbers in
{doc}`../../examples/generated/multislice-hrtem`. The lens optics (CTF, Scherzer, NCSI) are in
{doc}`35_hrem_simulation`.

In [ ]:
import math
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np

from pytex.app.phases import builtin_phase
from pytex.diffraction.hrem import AtomicSnapshot, MicroscopeAberrations
from pytex.diffraction.multislice import (
    MultisliceGrid,
    SlicedPotential,
    TemporalCoherence,
    multislice,
    parametrized_electron_scattering_factor,
    periodic_slab,
    zone_axis_cell,
)

np.set_printoptions(precision=4, suppress=True)
SILICON = builtin_phase("si_diamond").to_phase()
TUNGSTEN = builtin_phase("w_bcc").to_phase()
TITANIUM = builtin_phase("ti_hcp").to_phase()


def show(ax, image, extent, title, cmap="gray"):
    ax.imshow(image, cmap=cmap, origin="lower", extent=(0, extent[0], 0, extent[1]))
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("x (Å)")
    ax.set_ylabel("y (Å)")

## 1. The atomic potential

The engine builds the potential from the **electron scattering factor** $f_e$ of each atom, the
Fourier transform of its electrostatic potential. Three parametrizations are offered: Lobato & Van
Dyck (2014, the default and abTEM's), Kirkland (2010), and the Mott–Bethe transform of PyTex's
X-ray table (the one the Bloch-wave solver uses). They agree closely for silicon; the heavy
tungsten atom scatters far more strongly.

In [ ]:
g = np.linspace(0.0, 3.0, 200)
fig, ax = plt.subplots(figsize=(6, 3.5))
for method, style in (("lobato", "-"), ("kirkland", "--"), ("mott_bethe", ":")):
    ax.plot(g, parametrized_electron_scattering_factor("Si", g, method), style, label=f"Si, {method}")
ax.plot(g, parametrized_electron_scattering_factor("W", g), label="W, lobato")
ax.set_xlabel("g (Å⁻¹)")
ax.set_ylabel("f_e (Å)")
ax.set_yscale("log")
ax.legend(fontsize=8)
plt.show()

The multislice cuts the specimen into slices of thickness $\Delta z$ and projects each atom into
the slice holding its centre. For silicon along [001] with $\Delta z = a/4$ every slice holds one
atomic layer, and the four distinct layers repeat — the engine recognises identical slices and
builds each only once. The mean of the projected potential over the box is the (independent-atom)
**mean inner potential**.

In [ ]:
slab, cell, repeats = periodic_slab(SILICON, (0, 0, 1), (2, 2), beam_repeats=10)
grid = MultisliceGrid.from_sampling((slab.cell[0, 0], slab.cell[1, 1]), 0.05)
sliced = SlicedPotential.from_snapshot(slab, grid, SILICON.lattice.a / 4)
print(f"{sliced.num_slices} slices, {sliced.unique_slice_count} distinct")
print(f"mean inner potential {sliced.mean_inner_potential_volt():.2f} V")

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
extent = grid.extent_angstrom
for n, ax in enumerate(axes):
    show(ax, sliced.projected_potential(n), extent, f"slice {n}: v_n (V Å)", cmap="viridis")
plt.tight_layout()
plt.show()

## 2. A crystal needs an exactly periodic box

The Fourier grid is periodic, so the calculation is for an infinite repetition of the box. If the
box is not a lattice period, every edge is a seam. `zone_axis_cell` finds three perpendicular
*lattice* vectors with the zone axis along the beam, and `periodic_slab` fills that box exactly.

In [ ]:
for zone in [(0, 0, 1), (1, 1, 0), (1, 1, 1), (1, 1, 2)]:
    print(zone_axis_cell(SILICON, zone).describe())
print(zone_axis_cell(TITANIUM, (0, 0, 1)).describe())

The older `AtomicSnapshot.from_phase` pads the atoms' bounding box with a margin instead, so its
box is *not* a lattice period. The consequence is visible in the diffraction pattern: in diamond
the (200) reflection is forbidden, and it stays dark in the periodic box but not in the padded one.

In [ ]:
a = SILICON.lattice.a
periodic, _, _ = periodic_slab(SILICON, (0, 0, 1), (1, 1), beam_repeats=8)
padded = AtomicSnapshot.from_phase(SILICON, supercell=(1, 1, 8), zone_axis=(0, 0, 1))
print(f"periodic box {periodic.cell[0, 0]:.3f} Å, padded box {padded.cell[0, 0]:.3f} Å, a = {a:.3f} Å")
for name, snap in (("periodic", periodic), ("padded", padded)):
    wave = multislice(snap, 200.0, sampling_angstrom=0.08, slice_thickness_angstrom=a)
    lx = snap.cell[0, 0]
    i200 = wave.beam_intensities(np.array([[2 / lx, 0.0]]), -1)[0]
    print(f"{name:>9}: intensity at the (200) position {i200:.2e}")

## 3. Sampling and the band limit

Multiplying the wave by the transmission function doubles its bandwidth, so both are band-limited
to two thirds of the Nyquist frequency to prevent aliasing. The largest scattering angle the grid
represents is $\alpha_{\max} = \lambda g_{\max}$, and whatever scatters beyond it is lost: the
**retained intensity** reported by every run is therefore a check on the sampling. Tungsten, a
strong scatterer, needs a fine grid.

In [ ]:
w_slab, _, _ = periodic_slab(TUNGSTEN, (0, 0, 1), (2, 2), thickness_angstrom=60.0)
for sampling in (0.2, 0.1, 0.05, 0.025):
    wave = multislice(w_slab, 200.0, sampling_angstrom=sampling, slice_thickness_angstrom=TUNGSTEN.lattice.a / 2)
    print(
        f"dx = {sampling:5.3f} Å: alpha_max = {wave.grid.max_scattering_angle_mrad(200.0):6.1f} mrad, "
        f"retained intensity {wave.retained_intensity[-1]:.4f}"
    )

## 4. The exit wave and its evolution with thickness

Silicon along [110] shows the famous "dumbbells": pairs of atom columns 1.36 Å apart. A single
multislice pass stores the wave at any number of depths. The amplitude concentrates on the atom
columns and oscillates with depth — **channelling** — which is the real-space face of the
**Pendellösung** exchange of intensity between the Bragg beams.

In [ ]:
si110, cell110, _ = periodic_slab(SILICON, (1, 1, 0), (2, 1), thickness_angstrom=400.0)
depths = np.arange(1, 53) * cell110.lengths_angstrom[2]
thick = multislice(si110, 200.0, sampling_angstrom=0.05, slice_thickness_angstrom=cell110.lengths_angstrom[2] / 4,
                   exit_depths_angstrom=depths)
print(thick.describe())

extent = thick.grid.extent_angstrom
fig, axes = plt.subplots(2, 4, figsize=(12, 5))
for column, target in enumerate((20.0, 80.0, 150.0, 300.0)):
    index = thick.depth_index(target)
    psi = thick.wave(index)
    show(axes[0, column], np.abs(psi), extent, f"|ψ| at {thick.depths_angstrom[index]:.0f} Å", cmap="magma")
    show(axes[1, column], np.angle(psi), extent, "phase", cmap="twilight")
plt.tight_layout()
plt.show()

In [ ]:
# Beam intensities against thickness. The box has x along [001] and y along [1-10], so a
# reflection hkl sits at g_x = l / a and g_y = (h - k) / (a sqrt 2).
a = SILICON.lattice.a
reflections = {"000": (0, 0, 0), "111": (1, 1, 1), "1-11": (1, -1, 1), "002 (forbidden)": (0, 0, 2),
               "2-20": (2, -2, 0), "004": (0, 0, 4)}
labels = list(reflections)
g = np.array([[l / a, (h - k) / (a * math.sqrt(2))] for h, k, l in reflections.values()])
intensity = thick.beam_intensities(g)
fig, ax = plt.subplots(figsize=(7, 3.5))
for column, name in enumerate(labels):
    ax.plot(thick.depths_angstrom, intensity[:, column], label=name)
ax.set_xlabel("thickness (Å)")
ax.set_ylabel("beam intensity")
ax.legend(fontsize=8)
plt.show()

## 5. The diffraction pattern, and beam tilt

The far-field pattern is the squared Fourier transform of the exit wave. At the exact zone axis it
has the symmetry of the projection; a small beam tilt breaks Friedel symmetry, because the Ewald
sphere now cuts the reciprocal lattice asymmetrically.

In [ ]:
si_thin, _, _ = periodic_slab(SILICON, (1, 1, 0), (4, 3), thickness_angstrom=150.0)
level = multislice(si_thin, 200.0, sampling_angstrom=0.08, slice_thickness_angstrom=1.92)
tilted = multislice(si_thin, 200.0, sampling_angstrom=0.08, slice_thickness_angstrom=1.92, tilt_mrad=(5.0, 0.0))
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, wave, title in ((axes[0], level, "zone axis"), (axes[1], tilted, "tilted 5 mrad along x")):
    pattern = wave.diffraction_pattern()
    nyq = 0.5 / np.array(wave.grid.sampling_angstrom)
    ax.imshow(np.log10(pattern + 1e-7), cmap="inferno", origin="lower", extent=(-nyq[0], nyq[0], -nyq[1], nyq[1]))
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("g_x (Å⁻¹)")
plt.show()

## 6. HRTEM images under three kinds of optics

The objective lens multiplies the exit-wave spectrum by $A\,E_s\,E_c\,e^{-i\chi}$. At 300 kV:
an **uncorrected** lens at Scherzer defocus cannot separate the dumbbells; a **Cs-corrected** lens
at small defocus can; **negative Cs imaging** (NCSI) with overfocus shows the atoms bright.

In [ ]:
si110_img, _, _ = periodic_slab(SILICON, (1, 1, 0), (3, 2), thickness_angstrom=50.0)
exit_300 = multislice(si110_img, 300.0, sampling_angstrom=0.05, slice_thickness_angstrom=1.92)
uncorrected = MicroscopeAberrations.conventional_tem(energy_kev=300.0, cs_mm=1.0)
lenses = {
    f"uncorrected, Scherzer Δf = {uncorrected.scherzer_defocus_angstrom:.0f} Å": replace(
        uncorrected, defocus_angstrom=uncorrected.scherzer_defocus_angstrom
    ),
    "Cs-corrected (Cs = 5 µm), Δf = -40 Å": MicroscopeAberrations.cs_corrected(energy_kev=300.0),
    "NCSI": MicroscopeAberrations.ncsi(energy_kev=300.0),
}
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (title, lens) in zip(axes, lenses.items()):
    show(ax, exit_300.image(lens), exit_300.grid.extent_angstrom, title)
plt.tight_layout()
plt.show()

## 7. A focal series from one exit wave

The exit wave does not depend on the lens, and changing the defocus is the same as propagating the
exit wave through free space. A through-focus series therefore needs **one** multislice run and one
pair of FFTs per image. `focal_series` returns the images and their RMS contrast; experimental focal
series are the input to exit-wave reconstruction, which inverts exactly this model.

In [ ]:
corrected = MicroscopeAberrations.cs_corrected(energy_kev=300.0)
defoci = np.arange(-200.0, 201.0, 50.0)
series = exit_300.focal_series(corrected, defoci)
print(series.describe())

fig, axes = plt.subplots(1, len(defoci), figsize=(2.0 * len(defoci), 2.4))
for ax, df, image in zip(axes, series.defoci_angstrom, series.images):
    ax.imshow(image, cmap="gray", origin="lower")
    ax.set_title(f"Δf = {df:.0f} Å", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

fine = exit_300.focal_series(corrected, np.linspace(-300.0, 300.0, 61))
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(fine.defoci_angstrom, fine.contrasts * 100)
ax.set_xlabel("defocus Δf (Å), negative = underfocus")
ax.set_ylabel("RMS contrast (%)")
plt.show()

## 8. The defocus–thickness map

Neither the thickness nor the defocus of an experimental image is known in advance. The standard
remedy is to simulate a tableau — thickness down, defocus across — and find the tile that matches.
`defocus_thickness_map` builds it from a single run with several exit depths.

In [ ]:
si_map, cell_map, _ = periodic_slab(SILICON, (1, 1, 0), (2, 1), thickness_angstrom=200.0)
thicknesses = [30.0, 70.0, 120.0, 200.0]
wave_map = multislice(si_map, 300.0, sampling_angstrom=0.06, slice_thickness_angstrom=1.92, exit_depths_angstrom=thicknesses)
tableau = wave_map.defocus_thickness_map(corrected, [-120.0, -60.0, 0.0, 60.0, 120.0])
print(tableau.describe())
rows, cols = tableau.images.shape[:2]
fig, axes = plt.subplots(rows, cols, figsize=(2.2 * cols, 1.5 * rows))
for i in range(rows):
    for j in range(cols):
        axes[i, j].imshow(tableau.images[i, j], cmap="gray", origin="lower")
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])
        if i == 0:
            axes[i, j].set_title(f"Δf = {tableau.defoci_angstrom[j]:.0f} Å", fontsize=8)
        if j == 0:
            axes[i, j].set_ylabel(f"t = {tableau.thicknesses_angstrom[i]:.0f} Å", fontsize=8)
plt.tight_layout()
plt.show()

## 9. Partial coherence: Frank's envelope versus focal integration

Chromatic aberration spreads the defocus with a standard deviation $\Delta$. The usual shortcut
multiplies the wave transfer by Frank's envelope $E_c(g)$, which is exact for the *linear* image
terms — each diffracted beam interfering with the transmitted one. Integrating the image over the
defocus spread is exact for every term. The two differ in the *non-linear* terms, the interference
between two diffracted beams: for two equivalent beams $\pm\mathbf g$ the defocus phase cancels, so
the focal spread does not damp their interference at all, while the envelope damps it by
$E_c(g)^2$. At a large spread the linear transfer beyond about 0.8 Å⁻¹ is damped hard while those
non-linear terms are not, so the shortcut fails even for a thin carbon film, and grows worse the
more strongly the specimen scatters. (The integration itself is checked in the test suite against
a brute-force average over thousands of defoci.)

In [ ]:
lens_spread = replace(MicroscopeAberrations.cs_corrected(energy_kev=300.0), focal_spread_angstrom=40.0)
carbon_film = AtomicSnapshot.amorphous_sample(species="C", density_g_cm3=2.0, dimensions_angstrom=(12.0, 12.0, 20.0), seed=5)
cases = {
    "amorphous C, 20 Å": carbon_film,
    "Si [110], one period": periodic_slab(SILICON, (1, 1, 0), (2, 2), beam_repeats=1)[0],
    "Si [110], 60 Å": periodic_slab(SILICON, (1, 1, 0), (2, 2), thickness_angstrom=60.0)[0],
    "W [001], 60 Å": periodic_slab(TUNGSTEN, (0, 0, 1), (4, 4), thickness_angstrom=60.0)[0],
}
profiles = {}
for name, snap in cases.items():
    wave = multislice(snap, 300.0, sampling_angstrom=0.05, slice_thickness_angstrom=1.5)
    quasi = wave.image(lens_spread)
    exact = wave.image(lens_spread, temporal_coherence=TemporalCoherence.FOCAL_INTEGRATION)
    gap = np.max(np.abs(quasi - exact)) / np.ptp(exact)
    print(f"{name:>22}: largest difference {gap * 100:5.1f} % of the image range")
    profiles[name] = (wave.grid, quasi, exact)

grid_si, quasi, exact = profiles["Si [110], 60 Å"]
x = np.arange(grid_si.shape[1]) * grid_si.sampling_angstrom[0]
row = int(np.argmax(exact.max(axis=1)))
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(x, quasi[row], label="Frank envelope (quasi-coherent)")
ax.plot(x, exact[row], label="focal integration (exact)")
ax.set_xlabel("x (Å)")
ax.set_ylabel("intensity")
ax.set_title("Si [110], 60 Å: one row through the dumbbells", fontsize=9)
ax.legend(fontsize=8)
plt.show()

## 10. Thermal diffuse scattering with frozen phonons

Atoms vibrate. The frozen-phonon model runs the multislice for several random snapshots of the
displaced atoms and averages the **intensities**. The diffraction pattern then acquires a diffuse
background between the Bragg spots; the images lose a little contrast at high frequency.

In [ ]:
si_ph, _, _ = periodic_slab(SILICON, (1, 1, 0), (3, 2), thickness_angstrom=100.0)
static = multislice(si_ph, 200.0, sampling_angstrom=0.08, slice_thickness_angstrom=1.92)
thermal = multislice(
    si_ph, 200.0, sampling_angstrom=0.08, slice_thickness_angstrom=1.92,
    frozen_phonon_sigma_angstrom=0.076, frozen_phonon_configurations=8, seed=1,
)
print(thermal.describe())
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, wave, title in ((axes[0], static, "static lattice"), (axes[1], thermal, "8 frozen phonons")):
    nyq = 0.5 / np.array(wave.grid.sampling_angstrom)
    ax.imshow(np.log10(wave.diffraction_pattern() + 1e-8), cmap="inferno", origin="lower",
              extent=(-nyq[0], nyq[0], -nyq[1], nyq[1]), vmin=-8)
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_title(title, fontsize=9)
plt.show()

## 11. Other specimens

**Hexagonal titanium along [0001]**, **a missing atom column** (a vacancy column through the whole
foil, the strongest version of a point defect), and **amorphous carbon**, whose power spectrum shows
the Thon rings of the contrast transfer function.

In [ ]:
ti, _, _ = periodic_slab(TITANIUM, (0, 0, 1), (3, 2), thickness_angstrom=50.0)
ti_image = multislice(ti, 300.0, sampling_angstrom=0.05, slice_thickness_angstrom=TITANIUM.lattice.c / 2).image(corrected)

column, _, _ = periodic_slab(SILICON, (1, 1, 0), (3, 2), thickness_angstrom=50.0)
xy = column.positions[:, :2]
centre = xy[np.argmin(np.linalg.norm(xy - xy.mean(axis=0), axis=1))]
keep = np.linalg.norm(xy - centre, axis=1) > 0.3
missing = AtomicSnapshot(
    species=tuple(s for s, k in zip(column.species, keep) if k),
    positions=column.positions[keep],
    cell=column.cell,
    label="Si [110] with one empty column",
)
missing_image = multislice(missing, 300.0, sampling_angstrom=0.05, slice_thickness_angstrom=1.92).image(corrected)

carbon = AtomicSnapshot.amorphous_sample(species="C", density_g_cm3=2.0, dimensions_angstrom=(40.0, 40.0, 60.0), seed=3)
defocused = MicroscopeAberrations(energy_kev=300.0, defocus_angstrom=-700.0, cs_mm=1.2, focal_spread_angstrom=30.0,
                                  convergence_semiangle_mrad=0.1)
carbon_wave = multislice(carbon, 300.0, sampling_angstrom=0.1, slice_thickness_angstrom=2.0)
carbon_image = carbon_wave.image(defocused)
spectrum = np.fft.fftshift(np.abs(np.fft.fft2(carbon_image - carbon_image.mean())) ** 2)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
show(axes[0], ti_image, ti.cell[:2, :2].diagonal(), "Ti [0001], Cs-corrected")
show(axes[1], missing_image, column.cell[:2, :2].diagonal(), "Si [110], one empty column")
nyq = 0.5 / carbon_wave.grid.sampling_angstrom[0]
axes[2].imshow(np.log10(spectrum + 1), cmap="gray", origin="lower", extent=(-nyq, nyq, -nyq, nyq))
axes[2].set_xlim(-1.5, 1.5)
axes[2].set_ylim(-1.5, 1.5)
axes[2].set_title("amorphous C: Thon rings", fontsize=9)
plt.tight_layout()
plt.show()

## 12. A check against Bloch waves

Multislice and the Bloch-wave method solve the same equation by different routes. Give both the
same (Mott–Bethe) potential and keep only the zero-order Laue zone — the projected potential of one
period — and they must agree. With one slice per period the multislice carries the *splitting
error* of a 5.4 Å slice; splitting the projected potential into eight thinner slices removes it.

In [ ]:
import scipy.fft

from pytex.core.lattice import ZoneAxis
from pytex.diffraction.dynamical import beam_set_for_zone, solve_bloch_waves
from pytex.diffraction.hrem import relativistic_interaction_parameter_inv_v_angstrom
from pytex.diffraction.multislice import antialias_aperture, fresnel_propagator, slice_potential

a = SILICON.lattice.a
cell_001, _, _ = periodic_slab(SILICON, (0, 0, 1), (1, 1))
grid_001 = MultisliceGrid((64, 64), (a, a))
v = slice_potential(cell_001.species, cell_001.positions[:, :2], grid_001, "mott_bethe")
sigma = relativistic_interaction_parameter_inv_v_angstrom(200.0)
hkl = [(0, 0, 0), (2, 2, 0), (4, 0, 0)]
periods = (10, 20, 37)


def projected_multislice(splits):
    tau = np.fft.ifft2(np.fft.fft2(np.exp(1j * sigma * v / splits)) * antialias_aperture(grid_001))
    propagator = fresnel_propagator(grid_001, 200.0, a / splits)
    psi = np.ones(grid_001.shape, dtype=complex)
    out = {}
    for step in range(37 * splits):
        psi = np.fft.ifft2(np.fft.fft2(psi * tau) * propagator)
        if (step + 1) % splits == 0 and (step + 1) // splits in periods:
            spectrum = np.fft.fft2(psi) / psi.size
            out[(step + 1) // splits] = [abs(spectrum[k, h]) ** 2 for h, k, _ in hkl]
    return out


one, eight = projected_multislice(1), projected_multislice(8)
beams = beam_set_for_zone(SILICON, ZoneAxis(indices=(0, 0, 1), phase=SILICON), beam_energy_kev=200.0,
                          max_index=24, g_max_inv_angstrom=3.9, max_excitation_error_inv_angstrom=0.6)
print(f"{'t (Å)':>7} {'beam':>5} {'1 slice/period':>15} {'8 slices/period':>16} {'Bloch':>8}")
for n in periods:
    bloch = solve_bloch_waves(beams, [[0.0, 0.0]], thickness_angstrom=n * a)
    for column, (h, k, l) in enumerate(hkl):
        print(f"{n * a:7.1f} {f'{h}{k}{l}':>5} {one[n][column]:15.4f} {eight[n][column]:16.4f} "
              f"{float(bloch.intensity_of((h, k, l))[0]):8.4f}")

## Where next

- In the workbench, the **HRTEM** panel runs the same engine: the *Micrograph* view for one image,
  and the *Focal / thickness series* view for a focal series and a defocus–thickness map.
- `pytex.adapters.abtem.simulate_hrem(..., engine="abtem")` runs abTEM itself on the same specimen
  and lens, for comparison.
- Every result object explains itself: `describe()` on a `MultisliceExitWave`, `FocalSeries` or
  `DefocusThicknessMap` reports the conventions, the sampling and the diagnostics.